In [1]:
"""
=============================================================================
Multi-Task BiGRU Pipeline for Parkinson's Disease Prediction
=============================================================================
Targets: NHY, GROUP, DEPRESSION_BINARY, DEPRESSION_SEVERITY
Features: 512 MRI + 32 Clinical = 544 features (fused late fusion)
Augmentation: SMOTE+ENN → 5x
Model: Bidirectional GRU with Self-Attention (multi-task heads)
Validation: 5-Fold Stratified Cross-Validation, 500 epochs
=============================================================================
"""

import os
import warnings
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import (
    accuracy_score, f1_score, precision_score, recall_score,
    roc_auc_score, cohen_kappa_score, confusion_matrix,
    classification_report, mean_absolute_error, mean_squared_error
)
from imblearn.combine import SMOTEENN
from imblearn.over_sampling import SMOTE

warnings.filterwarnings('ignore')
np.random.seed(42)
torch.manual_seed(42)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(42)

# ========================== CONFIG ==========================
DATA_PATH = '/kaggle/input/datasets/bushraurooj21/depression/DEPRESSION.csv'
OUTPUT_DIR = '/kaggle/working/'  # Kaggle output
os.makedirs(OUTPUT_DIR, exist_ok=True)

TARGET_COLS = ['DEPRESSION_SEVERITY',"DEPRESSION_BINARY"]
ID_COL = 'PATNO'
N_FOLDS = 5
EPOCHS = 500
BATCH_SIZE = 32
LEARNING_RATE = 1e-3
PATIENCE = 50           # Early stopping patience
HIDDEN_SIZE = 128
NUM_LAYERS = 2
DROPOUT = 0.4
WEIGHT_DECAY = 1e-4
AUGMENTATION_FACTOR = 5  # Target 5x augmentation
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

print(f"Using device: {DEVICE}")

# ========================== 1. LOAD & INSPECT DATA ==========================
print("\n" + "="*70)
print("STEP 1: Loading and Inspecting Data")
print("="*70)

df = pd.read_csv(DATA_PATH)
print(f"Original shape: {df.shape}")
print(f"Columns: {df.columns.tolist()[:20]}... (showing first 20)")
print(f"\nTarget columns found: {[c for c in TARGET_COLS if c in df.columns]}")
print(f"ID column found: {ID_COL in df.columns}")

# Display target distributions
for col in TARGET_COLS:
    if col in df.columns:
        print(f"\n{col} distribution:")
        print(df[col].value_counts().sort_index())

# ========================== 2. PREPROCESSING ==========================
print("\n" + "="*70)
print("STEP 2: Preprocessing")
print("="*70)

# Drop ID column
if ID_COL in df.columns:
    df = df.drop(columns=[ID_COL])

# Separate features and targets
feature_cols = [c for c in df.columns if c not in TARGET_COLS]
X = df[feature_cols].values.astype(np.float32)
print(f"Feature matrix shape: {X.shape}")

# Handle NaN in features
from sklearn.impute import SimpleImputer
imputer = SimpleImputer(strategy='median')
X = imputer.fit_transform(X)

# Encode targets
label_encoders = {}
y_dict = {}
task_types = {}  # 'clf' or 'reg'
num_classes = {}

for col in TARGET_COLS:
    if col in df.columns:
        vals = df[col].copy()
        # Fill missing targets with mode
        vals = vals.fillna(vals.mode()[0])

        le = LabelEncoder()
        y_encoded = le.fit_transform(vals.astype(str))
        label_encoders[col] = le
        y_dict[col] = y_encoded
        n_cls = len(le.classes_)
        num_classes[col] = n_cls
        task_types[col] = 'clf'  # All treated as classification
        print(f"  {col}: {n_cls} classes -> {le.classes_}")

# Stack targets
targets_available = [col for col in TARGET_COLS if col in df.columns]
y_all = np.column_stack([y_dict[col] for col in targets_available])
print(f"Target matrix shape: {y_all.shape}")

# ========================== 3. SMOTE+ENN AUGMENTATION ==========================
print("\n" + "="*70)
print("STEP 3: SMOTE+ENN Data Augmentation (target ~5x)")
print("="*70)

# Create composite label for stratification (combine all targets into single label)
composite_labels = []
for i in range(len(y_all)):
    composite_labels.append("_".join([str(y_all[i, j]) for j in range(y_all.shape[1])]))
composite_labels = np.array(composite_labels)

# Encode composite labels
le_composite = LabelEncoder()
y_composite = le_composite.fit_transform(composite_labels)
print(f"Unique composite classes: {len(le_composite.classes_)}")
print(f"Composite class distribution: {Counter(y_composite)}")

# Iterative SMOTE+ENN augmentation to reach ~5x
# For very rare classes, we use SMOTE with adjusted k_neighbors
original_size = len(X)
target_size = original_size * AUGMENTATION_FACTOR

X_aug = X.copy()
y_all_aug = y_all.copy()

print(f"\nOriginal samples: {original_size}")
print(f"Target samples:   {target_size}")

# Apply SMOTE+ENN per-target to handle each target's imbalance
# Then combine. We use the most imbalanced target for primary augmentation.

# Strategy: Apply SMOTE on composite label
min_class_count = min(Counter(y_composite).values())
k_neighbors_smote = min(5, min_class_count - 1) if min_class_count > 1 else 1

try:
    # First try SMOTE+ENN
    smote_enn = SMOTEENN(
        smote=SMOTE(
            sampling_strategy='auto',
            k_neighbors=max(1, k_neighbors_smote),
            random_state=42
        ),
        random_state=42
    )
    X_resampled, y_composite_resampled = smote_enn.fit_resample(X_aug, y_composite)

    # If SMOTE+ENN doesn't give enough, apply SMOTE again
    if len(X_resampled) < target_size:
        # Calculate how much more we need
        current_counts = Counter(y_composite_resampled)
        max_count = max(current_counts.values())
        # Increase all classes to reach target
        target_per_class = int(target_size / len(current_counts))
        sampling_strategy = {
            cls: max(count, target_per_class)
            for cls, count in current_counts.items()
        }

        smote_extra = SMOTE(
            sampling_strategy=sampling_strategy,
            k_neighbors=max(1, k_neighbors_smote),
            random_state=42
        )
        X_resampled, y_composite_resampled = smote_extra.fit_resample(
            X_resampled, y_composite_resampled
        )

    # Decode composite labels back to individual targets
    composite_decoded = le_composite.inverse_transform(y_composite_resampled)
    y_all_aug = np.array([
        [int(x) for x in label.split("_")]
        for label in composite_decoded
    ])
    X_aug = X_resampled

    print(f"After SMOTE+ENN augmentation: {len(X_aug)} samples")
    print(f"Augmentation ratio: {len(X_aug)/original_size:.1f}x")

except Exception as e:
    print(f"SMOTE+ENN failed ({e}), trying per-target SMOTE...")
    # Fallback: per-target SMOTE
    X_aug_list = [X.copy()]
    y_aug_list = [y_all.copy()]

    for t_idx, col in enumerate(targets_available):
        y_t = y_all[:, t_idx]
        min_c = min(Counter(y_t).values())
        k = min(3, min_c - 1) if min_c > 1 else 1
        try:
            smote = SMOTE(k_neighbors=max(1, k), random_state=42 + t_idx)
            X_res, y_res = smote.fit_resample(X, y_t)
            # Reconstruct other targets via nearest neighbor
            from sklearn.neighbors import KNeighborsClassifier
            y_other = np.delete(y_all, t_idx, axis=1)
            knn = KNeighborsClassifier(n_neighbors=max(1, k))
            knn.fit(X, y_other if y_other.ndim > 1 else y_other.reshape(-1, 1))

            new_mask = np.arange(len(X), len(X_res))
            if len(new_mask) > 0:
                X_new = X_res[new_mask]
                y_other_pred = knn.predict(X_new)
                y_new_full = np.zeros((len(X_new), y_all.shape[1]), dtype=int)
                y_new_full[:, t_idx] = y_res[new_mask]
                other_cols = [i for i in range(y_all.shape[1]) if i != t_idx]
                if y_other_pred.ndim == 1:
                    y_new_full[:, other_cols[0]] = y_other_pred
                else:
                    for j, oc in enumerate(other_cols):
                        y_new_full[:, oc] = y_other_pred[:, j]
                X_aug_list.append(X_new)
                y_aug_list.append(y_new_full)
        except Exception as e2:
            print(f"  SMOTE failed for {col}: {e2}")
            continue

    X_aug = np.vstack(X_aug_list)
    y_all_aug = np.vstack(y_aug_list)

    # If still not enough, duplicate with noise
    while len(X_aug) < target_size:
        noise_idx = np.random.choice(len(X), min(original_size, target_size - len(X_aug)))
        X_noise = X[noise_idx] + np.random.normal(0, 0.01, X[noise_idx].shape).astype(np.float32)
        y_noise = y_all[noise_idx]
        X_aug = np.vstack([X_aug, X_noise])
        y_all_aug = np.vstack([y_all_aug, y_noise])

    print(f"After augmentation: {len(X_aug)} samples ({len(X_aug)/original_size:.1f}x)")

# Print augmented distributions
for t_idx, col in enumerate(targets_available):
    print(f"\n  {col} augmented distribution:")
    print(f"    {Counter(y_all_aug[:, t_idx])}")

# Save augmented distribution plot
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('Target Distributions After SMOTE+ENN Augmentation', fontsize=16, fontweight='bold')
for idx, col in enumerate(targets_available[:4]):
    ax = axes[idx // 2, idx % 2]
    counts = Counter(y_all_aug[:, idx])
    classes = sorted(counts.keys())
    vals = [counts[c] for c in classes]
    class_names = label_encoders[col].inverse_transform(classes)
    bars = ax.bar(range(len(classes)), vals, color=plt.cm.Set2(np.linspace(0, 1, len(classes))))
    ax.set_xticks(range(len(classes)))
    ax.set_xticklabels(class_names, rotation=45, ha='right')
    ax.set_title(f'{col}', fontsize=13, fontweight='bold')
    ax.set_ylabel('Count')
    for bar, v in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 2, str(v),
                ha='center', va='bottom', fontsize=10)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'augmented_distributions.png'), dpi=150, bbox_inches='tight')
plt.close()
print("\nSaved: augmented_distributions.png")


# ========================== 4. MODEL DEFINITION ==========================
print("\n" + "="*70)
print("STEP 4: Multi-Task BiGRU with Self-Attention")
print("="*70)


class SelfAttention(nn.Module):
    """Scaled dot-product self-attention over sequence dimension."""
    def __init__(self, hidden_size):
        super().__init__()
        self.query = nn.Linear(hidden_size, hidden_size)
        self.key = nn.Linear(hidden_size, hidden_size)
        self.value = nn.Linear(hidden_size, hidden_size)
        self.scale = np.sqrt(hidden_size)
        self.layer_norm = nn.LayerNorm(hidden_size)

    def forward(self, x):
        # x: (batch, seq_len, hidden)
        Q = self.query(x)
        K = self.key(x)
        V = self.value(x)

        attn_weights = torch.bmm(Q, K.transpose(1, 2)) / self.scale
        attn_weights = torch.softmax(attn_weights, dim=-1)
        context = torch.bmm(attn_weights, V)
        out = self.layer_norm(context + x)  # Residual connection
        return out, attn_weights


class MultiTaskBiGRU(nn.Module):
    """
    Bidirectional GRU with Self-Attention + Multi-Task Heads.
    Input: (batch, features) -> reshaped to (batch, seq_len, input_dim)
    """
    def __init__(self, input_dim, hidden_size, num_layers, dropout,
                 num_classes_dict, seq_len=1):
        super().__init__()
        self.seq_len = seq_len
        self.input_per_step = input_dim // seq_len

        # Shared BiGRU encoder
        self.bigru = nn.GRU(
            input_size=self.input_per_step,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
            bidirectional=True,
            dropout=dropout if num_layers > 1 else 0
        )

        self.attention = SelfAttention(hidden_size * 2)  # *2 for bidirectional

        # Batch normalization after attention
        self.bn = nn.BatchNorm1d(hidden_size * 2)
        self.dropout = nn.Dropout(dropout)

        # Task-specific heads with their own dropout & BN
        self.task_heads = nn.ModuleDict()
        for task_name, n_cls in num_classes_dict.items():
            self.task_heads[task_name] = nn.Sequential(
                nn.Linear(hidden_size * 2, hidden_size),
                nn.BatchNorm1d(hidden_size),
                nn.GELU(),
                nn.Dropout(dropout),
                nn.Linear(hidden_size, hidden_size // 2),
                nn.BatchNorm1d(hidden_size // 2),
                nn.GELU(),
                nn.Dropout(dropout * 0.5),
                nn.Linear(hidden_size // 2, n_cls)
            )

    def forward(self, x):
        batch_size = x.size(0)

        # Reshape flat features into sequence: (batch, seq_len, features_per_step)
        # We split 544 features into segments to create a pseudo-sequence
        x = x.view(batch_size, self.seq_len, self.input_per_step)

        # BiGRU encoding
        gru_out, _ = self.bigru(x)  # (batch, seq_len, hidden*2)

        # Self-attention
        attn_out, attn_weights = self.attention(gru_out)  # (batch, seq_len, hidden*2)

        # Global average pooling over sequence
        pooled = attn_out.mean(dim=1)  # (batch, hidden*2)

        # Batch norm + dropout on shared representation
        pooled = self.bn(pooled)
        pooled = self.dropout(pooled)

        # Task-specific predictions
        outputs = {}
        for task_name, head in self.task_heads.items():
            outputs[task_name] = head(pooled)

        return outputs, attn_weights


# ========================== 5. TRAINING UTILITIES ==========================

class FocalLoss(nn.Module):
    """Focal Loss for handling class imbalance."""
    def __init__(self, alpha=None, gamma=2.0, reduction='mean'):
        super().__init__()
        self.gamma = gamma
        self.alpha = alpha
        self.reduction = reduction

    def forward(self, inputs, targets):
        ce_loss = nn.functional.cross_entropy(inputs, targets, weight=self.alpha, reduction='none')
        pt = torch.exp(-ce_loss)
        focal_loss = ((1 - pt) ** self.gamma) * ce_loss
        if self.reduction == 'mean':
            return focal_loss.mean()
        return focal_loss.sum()


class UncertaintyWeighting(nn.Module):
    """Learnable task uncertainty weighting (Kendall et al., 2018)."""
    def __init__(self, n_tasks):
        super().__init__()
        self.log_vars = nn.Parameter(torch.zeros(n_tasks))

    def forward(self, losses):
        total = 0
        weighted_losses = []
        for i, loss in enumerate(losses):
            precision = torch.exp(-self.log_vars[i])
            weighted = precision * loss + self.log_vars[i]
            total += weighted
            weighted_losses.append(weighted.item())
        return total, weighted_losses


def compute_class_weights(y, device):
    """Compute inverse frequency class weights."""
    counts = np.bincount(y)
    counts = np.maximum(counts, 1)  # Avoid division by zero
    weights = 1.0 / counts
    weights = weights / weights.sum() * len(counts)
    return torch.FloatTensor(weights).to(device)


# ========================== 6. TRAINING & EVALUATION ==========================

def train_one_epoch(model, loader, optimizer, criterion_dict, uw, device):
    model.train()
    total_loss = 0
    all_preds = {t: [] for t in targets_available}
    all_labels = {t: [] for t in targets_available}

    for X_batch, *y_batches in loader:
        X_batch = X_batch.to(device)
        y_batch_dict = {
            targets_available[i]: y_batches[i].to(device)
            for i in range(len(targets_available))
        }

        optimizer.zero_grad()
        outputs, _ = model(X_batch)

        losses = []
        for t_idx, col in enumerate(targets_available):
            loss = criterion_dict[col](outputs[col], y_batch_dict[col])
            losses.append(loss)

            preds = outputs[col].argmax(dim=1).cpu().numpy()
            all_preds[col].extend(preds)
            all_labels[col].extend(y_batch_dict[col].cpu().numpy())

        total, _ = uw(losses)
        total.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        total_loss += total.item()

    # Compute per-task accuracy
    accs = {}
    for col in targets_available:
        accs[col] = accuracy_score(all_labels[col], all_preds[col])

    return total_loss / len(loader), accs


def evaluate(model, loader, criterion_dict, uw, device):
    model.eval()
    total_loss = 0
    all_preds = {t: [] for t in targets_available}
    all_labels = {t: [] for t in targets_available}
    all_probs = {t: [] for t in targets_available}

    with torch.no_grad():
        for X_batch, *y_batches in loader:
            X_batch = X_batch.to(device)
            y_batch_dict = {
                targets_available[i]: y_batches[i].to(device)
                for i in range(len(targets_available))
            }

            outputs, _ = model(X_batch)

            losses = []
            for t_idx, col in enumerate(targets_available):
                loss = criterion_dict[col](outputs[col], y_batch_dict[col])
                losses.append(loss)

                probs = torch.softmax(outputs[col], dim=1).cpu().numpy()
                preds = outputs[col].argmax(dim=1).cpu().numpy()
                all_preds[col].extend(preds)
                all_labels[col].extend(y_batch_dict[col].cpu().numpy())
                all_probs[col].extend(probs)

            total, _ = uw(losses)
            total_loss += total.item()

    accs = {}
    for col in targets_available:
        accs[col] = accuracy_score(all_labels[col], all_preds[col])

    return total_loss / len(loader), accs, all_preds, all_labels, all_probs


# ========================== 7. MAIN 5-FOLD CV LOOP ==========================
print("\n" + "="*70)
print("STEP 5: 5-Fold Cross-Validation Training (500 epochs)")
print("="*70)

# Determine sequence length for BiGRU (split features into segments)
n_features = X_aug.shape[1]
# Choose seq_len so that input_per_step is reasonable
# 544 features -> seq_len=16, input_per_step=34 OR seq_len=8, input_per_step=68
possible_seq_lens = [s for s in [32, 16, 8, 4, 2, 1] if n_features % s == 0]
SEQ_LEN = possible_seq_lens[0] if possible_seq_lens else 1
print(f"Features: {n_features}, Sequence length: {SEQ_LEN}, Features/step: {n_features // SEQ_LEN}")

# Create composite label for stratified splitting
composite_aug = []
for i in range(len(y_all_aug)):
    composite_aug.append("_".join([str(y_all_aug[i, j]) for j in range(y_all_aug.shape[1])]))
composite_aug = np.array(composite_aug)
le_strat = LabelEncoder()
y_strat = le_strat.fit_transform(composite_aug)

# Storage for all fold results
fold_results = {col: {
    'accuracy': [], 'f1_macro': [], 'f1_weighted': [],
    'precision_macro': [], 'recall_macro': [], 'kappa': [],
    'auc': [], 'best_epoch': [],
    'train_losses': [], 'val_losses': [],
    'train_accs': [], 'val_accs': [],
    'confusion_matrices': [], 'classification_reports': []
} for col in targets_available}

skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=42)

for fold, (train_idx, val_idx) in enumerate(skf.split(X_aug, y_strat)):
    print(f"\n{'='*50}")
    print(f"FOLD {fold + 1}/{N_FOLDS}")
    print(f"{'='*50}")
    print(f"Train: {len(train_idx)}, Val: {len(val_idx)}")

    # Split
    X_train, X_val = X_aug[train_idx], X_aug[val_idx]
    y_train = {col: y_all_aug[train_idx, i] for i, col in enumerate(targets_available)}
    y_val = {col: y_all_aug[val_idx, i] for i, col in enumerate(targets_available)}

    # Scale features
    scaler = StandardScaler()
    X_train_s = scaler.fit_transform(X_train).astype(np.float32)
    X_val_s = scaler.transform(X_val).astype(np.float32)

    # Create DataLoaders
    train_tensors = [torch.FloatTensor(X_train_s)] + [
        torch.LongTensor(y_train[col]) for col in targets_available
    ]
    val_tensors = [torch.FloatTensor(X_val_s)] + [
        torch.LongTensor(y_val[col]) for col in targets_available
    ]

    train_dataset = TensorDataset(*train_tensors)
    val_dataset = TensorDataset(*val_tensors)

    train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, drop_last=False)
    val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)

    # Build model
    model = MultiTaskBiGRU(
        input_dim=n_features,
        hidden_size=HIDDEN_SIZE,
        num_layers=NUM_LAYERS,
        dropout=DROPOUT,
        num_classes_dict={col: num_classes[col] for col in targets_available},
        seq_len=SEQ_LEN
    ).to(DEVICE)

    # Loss functions with class weights + focal loss
    criterion_dict = {}
    for col in targets_available:
        weights = compute_class_weights(y_train[col], DEVICE)
        criterion_dict[col] = FocalLoss(alpha=weights, gamma=2.0)

    # Uncertainty weighting
    uw = UncertaintyWeighting(len(targets_available)).to(DEVICE)

    # Optimizer with weight decay
    optimizer = optim.AdamW(
        list(model.parameters()) + list(uw.parameters()),
        lr=LEARNING_RATE,
        weight_decay=WEIGHT_DECAY
    )

    # Cosine annealing scheduler
    scheduler = optim.lr_scheduler.CosineAnnealingWarmRestarts(
        optimizer, T_0=50, T_mult=2, eta_min=1e-6
    )

    # Training loop
    best_val_loss = float('inf')
    best_model_state = None
    patience_counter = 0
    train_losses, val_losses = [], []
    train_accs_hist = {col: [] for col in targets_available}
    val_accs_hist = {col: [] for col in targets_available}

    for epoch in range(EPOCHS):
        train_loss, train_acc = train_one_epoch(
            model, train_loader, optimizer, criterion_dict, uw, DEVICE
        )
        val_loss, val_acc, _, _, _ = evaluate(
            model, val_loader, criterion_dict, uw, DEVICE
        )
        scheduler.step()

        train_losses.append(train_loss)
        val_losses.append(val_loss)
        for col in targets_available:
            train_accs_hist[col].append(train_acc[col])
            val_accs_hist[col].append(val_acc[col])

        # Early stopping on validation loss
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_model_state = {k: v.clone() for k, v in model.state_dict().items()}
            best_epoch = epoch + 1
            patience_counter = 0
        else:
            patience_counter += 1

        if (epoch + 1) % 50 == 0:
            avg_train = np.mean([train_acc[c] for c in targets_available])
            avg_val = np.mean([val_acc[c] for c in targets_available])
            print(f"  Epoch {epoch+1}/{EPOCHS} | "
                  f"Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f} | "
                  f"Avg Train Acc: {avg_train:.4f} | Avg Val Acc: {avg_val:.4f}")

        if patience_counter >= PATIENCE:
            print(f"  Early stopping at epoch {epoch + 1} (best: {best_epoch})")
            break

    # Load best model
    model.load_state_dict(best_model_state)
    _, _, val_preds, val_labels, val_probs = evaluate(
        model, val_loader, criterion_dict, uw, DEVICE
    )

    # Store results per task
    for t_idx, col in enumerate(targets_available):
        y_true = np.array(val_labels[col])
        y_pred = np.array(val_preds[col])
        y_prob = np.array(val_probs[col])

        acc = accuracy_score(y_true, y_pred)
        f1_mac = f1_score(y_true, y_pred, average='macro', zero_division=0)
        f1_w = f1_score(y_true, y_pred, average='weighted', zero_division=0)
        prec = precision_score(y_true, y_pred, average='macro', zero_division=0)
        rec = recall_score(y_true, y_pred, average='macro', zero_division=0)
        kappa = cohen_kappa_score(y_true, y_pred)

        # AUC (OvR)
        try:
            if num_classes[col] == 2:
                auc = roc_auc_score(y_true, y_prob[:, 1])
            else:
                from sklearn.preprocessing import label_binarize
                y_bin = label_binarize(y_true, classes=list(range(num_classes[col])))
                auc = roc_auc_score(y_bin, y_prob, multi_class='ovr', average='macro')
        except:
            auc = 0.0

        cm = confusion_matrix(y_true, y_pred)
        cr = classification_report(
            y_true, y_pred,
            target_names=label_encoders[col].classes_,
            zero_division=0, output_dict=True
        )

        fold_results[col]['accuracy'].append(acc)
        fold_results[col]['f1_macro'].append(f1_mac)
        fold_results[col]['f1_weighted'].append(f1_w)
        fold_results[col]['precision_macro'].append(prec)
        fold_results[col]['recall_macro'].append(rec)
        fold_results[col]['kappa'].append(kappa)
        fold_results[col]['auc'].append(auc)
        fold_results[col]['best_epoch'].append(best_epoch)
        fold_results[col]['train_losses'].append(train_losses)
        fold_results[col]['val_losses'].append(val_losses)
        fold_results[col]['train_accs'].append(train_accs_hist[col])
        fold_results[col]['val_accs'].append(val_accs_hist[col])
        fold_results[col]['confusion_matrices'].append(cm)
        fold_results[col]['classification_reports'].append(cr)

        print(f"\n  {col}: Acc={acc:.4f} | F1={f1_mac:.4f} | AUC={auc:.4f} | Kappa={kappa:.4f}")

    # Save model for this fold
    torch.save(best_model_state, os.path.join(OUTPUT_DIR, f'model_fold{fold+1}.pt'))


# ========================== 8. AGGREGATE & SAVE RESULTS ==========================
print("\n" + "="*70)
print("STEP 6: Aggregating Results Across Folds")
print("="*70)

results_summary = []
for col in targets_available:
    row = {
        'Target': col,
        'Accuracy': f"{np.mean(fold_results[col]['accuracy']):.4f} ± {np.std(fold_results[col]['accuracy']):.4f}",
        'F1_Macro': f"{np.mean(fold_results[col]['f1_macro']):.4f} ± {np.std(fold_results[col]['f1_macro']):.4f}",
        'F1_Weighted': f"{np.mean(fold_results[col]['f1_weighted']):.4f} ± {np.std(fold_results[col]['f1_weighted']):.4f}",
        'Precision': f"{np.mean(fold_results[col]['precision_macro']):.4f} ± {np.std(fold_results[col]['precision_macro']):.4f}",
        'Recall': f"{np.mean(fold_results[col]['recall_macro']):.4f} ± {np.std(fold_results[col]['recall_macro']):.4f}",
        'AUC-ROC': f"{np.mean(fold_results[col]['auc']):.4f} ± {np.std(fold_results[col]['auc']):.4f}",
        'Kappa': f"{np.mean(fold_results[col]['kappa']):.4f} ± {np.std(fold_results[col]['kappa']):.4f}",
        'Avg_Accuracy_Raw': np.mean(fold_results[col]['accuracy']),
        'Avg_F1_Raw': np.mean(fold_results[col]['f1_macro']),
    }
    results_summary.append(row)
    print(f"\n{col}:")
    for k, v in row.items():
        if k not in ['Target', 'Avg_Accuracy_Raw', 'Avg_F1_Raw']:
            print(f"  {k}: {v}")

df_results = pd.DataFrame(results_summary)
df_results.to_csv(os.path.join(OUTPUT_DIR, 'results_summary.csv'), index=False)
print("\nSaved: results_summary.csv")


# ========================== 9. VISUALIZATIONS ==========================
print("\n" + "="*70)
print("STEP 7: Generating Visualizations")
print("="*70)

# ---- 9a. Training/Validation Loss Curves (per fold, per target) ----
fig, axes = plt.subplots(N_FOLDS, 1, figsize=(14, 4 * N_FOLDS))
if N_FOLDS == 1:
    axes = [axes]
for fold in range(N_FOLDS):
    ax = axes[fold]
    t_loss = fold_results[targets_available[0]]['train_losses'][fold]
    v_loss = fold_results[targets_available[0]]['val_losses'][fold]
    ax.plot(t_loss, label='Train Loss', color='#2196F3', linewidth=1.5)
    ax.plot(v_loss, label='Val Loss', color='#F44336', linewidth=1.5)
    best_ep = fold_results[targets_available[0]]['best_epoch'][fold]
    ax.axvline(x=best_ep, color='green', linestyle='--', alpha=0.7, label=f'Best Epoch ({best_ep})')
    ax.set_title(f'Fold {fold + 1} - Loss Curves', fontsize=13, fontweight='bold')
    ax.set_xlabel('Epoch')
    ax.set_ylabel('Loss')
    ax.legend()
    ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'loss_curves_all_folds.png'), dpi=150, bbox_inches='tight')
plt.close()
print("Saved: loss_curves_all_folds.png")

# ---- 9b. Accuracy Curves per Target ----
for col in targets_available:
    fig, axes = plt.subplots(1, N_FOLDS, figsize=(5 * N_FOLDS, 4))
    if N_FOLDS == 1:
        axes = [axes]
    for fold in range(N_FOLDS):
        ax = axes[fold]
        ax.plot(fold_results[col]['train_accs'][fold], label='Train', color='#2196F3', linewidth=1.2)
        ax.plot(fold_results[col]['val_accs'][fold], label='Val', color='#F44336', linewidth=1.2)
        ax.set_title(f'Fold {fold+1}', fontsize=11)
        ax.set_xlabel('Epoch')
        ax.set_ylabel('Accuracy')
        ax.legend(fontsize=8)
        ax.grid(True, alpha=0.3)
        ax.set_ylim(0, 1.05)
    fig.suptitle(f'{col} - Accuracy Curves', fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, f'accuracy_curves_{col}.png'), dpi=150, bbox_inches='tight')
    plt.close()
print("Saved: accuracy_curves_<target>.png")

# ---- 9c. Confusion Matrices (last fold or averaged) ----
for col in targets_available:
    fig, axes = plt.subplots(1, N_FOLDS, figsize=(5 * N_FOLDS, 4.5))
    if N_FOLDS == 1:
        axes = [axes]
    class_names = label_encoders[col].classes_
    for fold in range(N_FOLDS):
        ax = axes[fold]
        cm = fold_results[col]['confusion_matrices'][fold]
        sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
                    xticklabels=class_names, yticklabels=class_names)
        ax.set_title(f'Fold {fold+1}', fontsize=11)
        ax.set_xlabel('Predicted')
        ax.set_ylabel('True')
    fig.suptitle(f'{col} - Confusion Matrices', fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, f'confusion_matrix_{col}.png'), dpi=150, bbox_inches='tight')
    plt.close()
print("Saved: confusion_matrix_<target>.png")

# ---- 9d. Per-Class F1 Scores Heatmap ----
for col in targets_available:
    class_names = label_encoders[col].classes_
    f1_data = np.zeros((N_FOLDS, len(class_names)))
    for fold in range(N_FOLDS):
        cr = fold_results[col]['classification_reports'][fold]
        for c_idx, cn in enumerate(class_names):
            if cn in cr:
                f1_data[fold, c_idx] = cr[cn]['f1-score']

    fig, ax = plt.subplots(figsize=(max(8, len(class_names) * 2), 5))
    sns.heatmap(f1_data, annot=True, fmt='.3f', cmap='YlOrRd',
                xticklabels=class_names, yticklabels=[f'Fold {i+1}' for i in range(N_FOLDS)], ax=ax)
    ax.set_title(f'{col} - Per-Class F1 Score Across Folds', fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, f'per_class_f1_{col}.png'), dpi=150, bbox_inches='tight')
    plt.close()
print("Saved: per_class_f1_<target>.png")

# ---- 9e. Overall Metrics Comparison Bar Chart ----
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
metrics_to_plot = ['Avg_Accuracy_Raw', 'Avg_F1_Raw']
titles = ['Average Accuracy', 'Average Macro-F1']
colors = ['#4CAF50', '#FF9800', '#2196F3', '#9C27B0']

for ax, metric, title in zip(axes, metrics_to_plot, titles):
    vals = [r[metric] for r in results_summary]
    names = [r['Target'] for r in results_summary]
    bars = ax.bar(names, vals, color=colors[:len(names)], edgecolor='black', linewidth=0.5)
    for bar, v in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
                f'{v:.4f}', ha='center', va='bottom', fontsize=11, fontweight='bold')
    ax.set_title(title, fontsize=14, fontweight='bold')
    ax.set_ylim(0, 1.1)
    ax.set_ylabel('Score')
    ax.grid(True, axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'overall_metrics_comparison.png'), dpi=150, bbox_inches='tight')
plt.close()
print("Saved: overall_metrics_comparison.png")

# ---- 9f. Radar Chart of All Metrics ----
metrics_radar = ['accuracy', 'f1_macro', 'precision_macro', 'recall_macro', 'kappa', 'auc']
labels_radar = ['Accuracy', 'F1-Macro', 'Precision', 'Recall', 'Kappa', 'AUC']

fig, ax = plt.subplots(figsize=(8, 8), subplot_kw=dict(projection='polar'))
angles = np.linspace(0, 2 * np.pi, len(labels_radar), endpoint=False).tolist()
angles += angles[:1]

for t_idx, col in enumerate(targets_available):
    values = [np.mean(fold_results[col][m]) for m in metrics_radar]
    values += values[:1]
    ax.plot(angles, values, 'o-', linewidth=2, label=col, color=colors[t_idx % len(colors)])
    ax.fill(angles, values, alpha=0.1, color=colors[t_idx % len(colors)])

ax.set_xticks(angles[:-1])
ax.set_xticklabels(labels_radar, fontsize=11)
ax.set_ylim(0, 1)
ax.set_title('Multi-Task Performance Radar', fontsize=14, fontweight='bold', pad=20)
ax.legend(loc='upper right', bbox_to_anchor=(1.3, 1.1))
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'radar_chart.png'), dpi=150, bbox_inches='tight')
plt.close()
print("Saved: radar_chart.png")

# ---- 9g. Box Plot of Metrics Across Folds ----
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
for idx, (metric, label) in enumerate(zip(metrics_radar, labels_radar)):
    ax = axes[idx // 3, idx % 3]
    data = [fold_results[col][metric] for col in targets_available]
    bp = ax.boxplot(data, labels=targets_available, patch_artist=True)
    for patch, color in zip(bp['boxes'], colors):
        patch.set_facecolor(color)
        patch.set_alpha(0.6)
    ax.set_title(label, fontsize=12, fontweight='bold')
    ax.set_ylabel('Score')
    ax.grid(True, alpha=0.3)
fig.suptitle('Metrics Distribution Across 5 Folds', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'boxplot_metrics.png'), dpi=150, bbox_inches='tight')
plt.close()
print("Saved: boxplot_metrics.png")

# ---- 9h. Detailed Classification Reports to CSV ----
for col in targets_available:
    all_reports = []
    for fold in range(N_FOLDS):
        cr = fold_results[col]['classification_reports'][fold]
        for cls_name, metrics in cr.items():
            if isinstance(metrics, dict):
                row = {'Fold': fold + 1, 'Class': cls_name}
                row.update(metrics)
                all_reports.append(row)
    df_cr = pd.DataFrame(all_reports)
    df_cr.to_csv(os.path.join(OUTPUT_DIR, f'classification_report_{col}.csv'), index=False)
print("Saved: classification_report_<target>.csv")


# ========================== 10. FINAL SUMMARY ==========================
print("\n" + "="*70)
print("FINAL SUMMARY")
print("="*70)

print(f"\nModel: Multi-Task BiGRU with Self-Attention")
print(f"Data: {original_size} original → {len(X_aug)} augmented ({len(X_aug)/original_size:.1f}x)")
print(f"Features: {n_features} (512 MRI + 32 Clinical)")
print(f"Sequence Length: {SEQ_LEN}, Features/Step: {n_features // SEQ_LEN}")
print(f"Hidden Size: {HIDDEN_SIZE}, Layers: {NUM_LAYERS}, Dropout: {DROPOUT}")
print(f"Epochs: {EPOCHS} (with early stopping, patience={PATIENCE})")
print(f"Cross-Validation: {N_FOLDS}-Fold Stratified")
print(f"Loss: Focal Loss + Uncertainty Weighting")
print(f"Optimizer: AdamW (lr={LEARNING_RATE}, wd={WEIGHT_DECAY})")
print(f"Scheduler: CosineAnnealingWarmRestarts")

print(f"\n{'Target':<25} {'Accuracy':<20} {'F1-Macro':<20} {'AUC-ROC':<20} {'Kappa':<20}")
print("-" * 105)
for col in targets_available:
    acc = f"{np.mean(fold_results[col]['accuracy']):.4f}±{np.std(fold_results[col]['accuracy']):.4f}"
    f1 = f"{np.mean(fold_results[col]['f1_macro']):.4f}±{np.std(fold_results[col]['f1_macro']):.4f}"
    auc = f"{np.mean(fold_results[col]['auc']):.4f}±{np.std(fold_results[col]['auc']):.4f}"
    kap = f"{np.mean(fold_results[col]['kappa']):.4f}±{np.std(fold_results[col]['kappa']):.4f}"
    print(f"{col:<25} {acc:<20} {f1:<20} {auc:<20} {kap:<20}")

print(f"\nAll outputs saved to: {OUTPUT_DIR}")
print("="*70)
print("PIPELINE COMPLETE")
print("="*70)

Using device: cuda

STEP 1: Loading and Inspecting Data
Original shape: (397, 547)
Columns: ['PATNO', 'MRI_0', 'MRI_1', 'MRI_2', 'MRI_3', 'MRI_4', 'MRI_5', 'MRI_6', 'MRI_7', 'MRI_8', 'MRI_9', 'MRI_10', 'MRI_11', 'MRI_12', 'MRI_13', 'MRI_14', 'MRI_15', 'MRI_16', 'MRI_17', 'MRI_18']... (showing first 20)

Target columns found: ['DEPRESSION_SEVERITY', 'DEPRESSION_BINARY']
ID column found: True

DEPRESSION_SEVERITY distribution:
DEPRESSION_SEVERITY
0    328
1     53
2     10
3      6
Name: count, dtype: int64

DEPRESSION_BINARY distribution:
DEPRESSION_BINARY
0    328
1     69
Name: count, dtype: int64

STEP 2: Preprocessing
Feature matrix shape: (397, 544)
  DEPRESSION_SEVERITY: 4 classes -> ['0' '1' '2' '3']
  DEPRESSION_BINARY: 2 classes -> ['0' '1']
Target matrix shape: (397, 2)

STEP 3: SMOTE+ENN Data Augmentation (target ~5x)
Unique composite classes: 4
Composite class distribution: Counter({np.int64(0): 328, np.int64(1): 53, np.int64(2): 10, np.int64(3): 6})

Original samples: 397
T